In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# Prepare and tokenize dataset
# dataset = load_dataset("yelp_review_full")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# def tokenize_function(examples):
#     return tokenizer(examples["text"], padding="max_length", truncation=True)

# tokenized_datasets = dataset.map(tokenize_function, batched=True)
# small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(200))
# small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(200))

# # Setup evaluation 
# metric = evaluate.load("accuracy")

# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = np.argmax(logits, axis=-1)
#     return metric.compute(predictions=predictions, references=labels)

# # Load pretrained model and evaluate model after each epoch
# model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)
# training_args = TrainingArguments(output_dir="test_trainer", evaluation_strategy="epoch")

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=small_train_dataset,
#     eval_dataset=small_eval_dataset,
#     compute_metrics=compute_metrics,
# )

# trainer.train()

/mnt/data_disk/chu123/anaconda3/envs/jyuzh/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


/mnt/data_disk/chu123/anaconda3/envs/jyuzh/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Map: 100%|██████████| 5427/5427 [00:00<00:00, 77087.29 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/mnt/data_disk/chu123/anaconda3/envs/jyuzh/lib/python3.12/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use 

RuntimeError: CUDA error: peer mapping resources exhausted
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [1]:
import os
import torch
import torch.nn as nn
from torch.nn import BCEWithLogitsLoss
from transformers import BertModel, BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType
import evaluate
import numpy as np
from transformers import DataCollatorWithPadding
# Convert labels to multi-label format
def convert_labels_to_multilabel(examples):
    labels = examples['labels']
    multi_labels = np.zeros(28)  # Assuming 28 emotion labels
    for i, label_list in enumerate(labels):
        for label in label_list:
            multi_labels[i][label] = 1
    examples['labels'] = multi_labels.tolist()
    return examples

# 数据预处理
def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length')

# 定义自定义的PyTorch模型类
class CustomBERTModel(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased', num_labels=28):
        super(CustomBERTModel, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
    
    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs[1]  # 获取池化的输出
        logits = self.classifier(pooled_output)
        return logits

/mnt/data_disk/chu123/anaconda3/envs/jyuzh/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")


In [1]:
import torch
import transformers
torch.__version__, transformers.__version__

/mnt/data_disk/chu123/anaconda3/envs/jyuzh/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


('2.4.0.post301', '4.44.1')

In [2]:
torch.cuda.is_available()

True

In [1]:
import pandas as pd
from datasets import Dataset

# 假设你已经有一个Pandas DataFrame
data = {
    'text': ["I love programming!", "I feel sad today."],
    'label': [1, 0]
}
df = pd.DataFrame(data)

# 将Pandas DataFrame转换为Hugging Face的Dataset对象
dataset = Dataset.from_pandas(df)

# 打印Dataset对象以验证转换
print(dataset)

/mnt/data_disk/chu123/anaconda3/envs/jyuzh/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['text', 'label'],
    num_rows: 2
})


In [4]:
import torch
import transformers
from transformers import BertTokenizer, BertPreTrainedModel
from bert_finetune import CustomBERTModel
# 使用微调后的模型进行预测
texts = [
    "I love programming!",
    "I feel sad today."
]
tokenizer = BertTokenizer.from_pretrained("model/fine_tuned_bert")
model = CustomBERTModel.from_pretrained("model/fine_tuned_bert")
# inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
# outputs = model(**inputs)
# predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

# # 打印预测结果
# for i, text in enumerate(texts):
#     print(f"Text: {text}")
#     print(f"Predicted probabilities: {predictions[i].detach().numpy()}")

/mnt/data_disk/chu123/anaconda3/envs/jyuzh/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BertConfig {
  "_name_or_path": "model/fine_tuned_bert",
  "architectures": [
    "CustomBERTModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "cls_num_labels": 28,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_name": "bert-base-uncased",
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.44.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

BertConfig {
  "_name_or_path": "model/fine_tuned_bert",
  "architectures": [
    "CustomBERTModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "cls_num_labels": 28,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  

In [32]:
model.cuda(1)
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
inputs = {k: v.cuda(1) for k, v in inputs.items()}
outputs = model(**inputs)

RuntimeError: Invalid device string: '1'

In [31]:
inputs

{'input_ids': tensor([[ 101, 1045, 2293, 4730,  999,  102,    0],
         [ 101, 1045, 2514, 6517, 2651, 1012,  102]], device='cuda:1'),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0]], device='cuda:1'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1, 1]], device='cuda:1')}

In [23]:
inputs['input_ids'].shape

torch.Size([2, 7])

In [15]:
lst = []
lst.extend(outputs['logits'].softmax(dim=-1)[:,4].tolist())

In [19]:
outputs['logits'].softmax(dim=-1)[:,4].tolist()

[0.002086709486320615, 0.002517945133149624]

In [19]:
model.classifier(outputs['pooler_output'])

AttributeError: 'BertModel' object has no attribute 'classifier'

In [28]:
from datasets import load_dataset
datasets = load_dataset("go_emotions", "simplified")
viewdata = datasets['train'].select(range(5))

In [32]:
viewdata[0]['labels']

AttributeError: 'list' object has no attribute 'id2str'

In [33]:
datasets.features

AttributeError: 'DatasetDict' object has no attribute 'features'

In [2]:
label_map = {
    0: 'admiration',
    1: 'amusement',
    2: 'anger',
    3: 'annoyance',
    4: 'approval',
    5: 'caring',
    6: 'confusion',
    7: 'curiosity',
    8: 'desire',
    9: 'disappointment',
    10: 'disapproval',
    11: 'disgust',
    12: 'embarrassment',
    13: 'excitement',
    14: 'fear',
    15: 'gratitude',
    16: 'grief',
    17: 'joy',
    18: 'love',
    19: 'nervousness',
    20: 'optimism',
    21: 'pride',
    22: 'realization',
    23: 'relief',
    24: 'remorse',
    25: 'sadness',
    26: 'surprise',
    27: 'neutral'
}

print(label_map.values())

dict_values(['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'])


In [1]:
import pandas as pd
import os
import itertools

# 读取文件夹中的所有文件
file_list = os.listdir('data')

# 筛选出前缀为 'Appliance_disappointment' 和 'Appliance_regression' 的文件
disappointment_files = [f for f in file_list if f.__contains__('disappointment')]
regression_files = [f for f in file_list if f.__contains__('regression')]
pair = [(i,j) for i,j in itertools.product(disappointment_files, regression_files) if i.split('_')[0] == j.split('_')[0]]
# pair = = [(i, [(i, j) j) for for i, i, j j in in itertools.product(disappointment_files, itertools.product(disappointment_files, regression_files) regression_files) if if i.split('_')[0] i.split('_')[0] == == j.split('_')[0]] j.split('_')[0]]
# disappointment_files, regression_files
# 读取并合并文件



In [3]:
for file1,file2 in pair:
    df1 = pd.read_csv(os.path.join('data', file1))
    print(df1.head())
    df2 = pd.read_csv(os.path.join('data', file2))
    print(df2.head())
    merged_df = pd.merge(df1,df2, on=['asin','parent_asin','user_id'])
    merged_df.to_csv(os.path.join('data', file1.split('_')[0] + '_merged.csv'), index=False)
    # break
# merged_df.head()

         asin parent_asin                       user_id  disappointment
0  B09BGPFTDB  B09BGPFTDB  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ        0.021089
1  0593235657  0593235657  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ        0.003102
2  1782490671  1782490671  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ        0.001153
3  0593138228  0593138228  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ        0.001506
4  0823098079  0823098079  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ        0.001676
   rating                                              title  \
0       1      Not a watercolor book! Seems like copies imo.   
1       5  Updated: after 1st arrived damaged this one is...   
2       5                              Excellent! I love it!   
3       5       Updated after 1st arrived damaged. Excellent   
4       5                                Beautiful patterns!   

                                                text  \
0  It is definitely not a watercolor book.  The p...   
1  Updated: after first book arrived very damaged...   
2  I bought it 

In [4]:
df1

,asin,parent_asin,user_id
0,B09BGPFTDB,B09BGPFTDB,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
1,0593235657,0593235657,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
2,1782490671,1782490671,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
3,0593138228,0593138228,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
4,0823098079,0823098079,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ
...,...,...,...
999995,B008SLDX20,B008SLDX20,AEMUH2AACMXFE7JPIJZ373GL2HIQ
999996,0440207622,0440207622,AEMUH2AACMXFE7JPIJZ373GL2HIQ
999997,044021422X,044021422X,AEMUH2AACMXFE7JPIJZ373GL2HIQ
999998,0440200563,0440200563,AEMUH2AACMXFE7JPIJZ373GL2HIQ


In [27]:
from jsonpath import jsonpath
import json
a = [[{'label': 'disappointment', 'score': 0.46669596433639526}, {'label': 'sadness', 'score': 0.3984946608543396}, {'label': 'annoyance', 'score': 0.06806609779596329}, {'label': 'neutral', 'score': 0.05703018978238106}, {'label': 'disapproval', 'score': 0.04423944652080536}, {'label': 'nervousness', 'score': 0.014850739389657974}, {'label': 'realization', 'score': 0.014059904962778091}, {'label': 'approval', 'score': 0.011267454363405704}, {'label': 'joy', 'score': 0.0063033816404640675}, {'label': 'remorse', 'score': 0.006221489980816841}, {'label': 'caring', 'score': 0.006029392126947641}, {'label': 'embarrassment', 'score': 0.005265495739877224}, {'label': 'anger', 'score': 0.004981442354619503}, {'label': 'disgust', 'score': 0.004259037785232067}, {'label': 'grief', 'score': 0.004002134781330824}, {'label': 'confusion', 'score': 0.0033829279709607363}, {'label': 'relief', 'score': 0.0031404944602400064}, {'label': 'desire', 'score': 0.00282747158780694}, {'label': 'admiration', 'score': 0.002815793501213193}, {'label': 'fear', 'score': 0.002707524225115776}, {'label': 'optimism', 'score': 0.0026164923328906298}, {'label': 'love', 'score': 0.0024883896112442017}, {'label': 'excitement', 'score': 0.0024494787212461233}, {'label': 'curiosity', 'score': 0.002374367555603385}, {'label': 'amusement', 'score': 0.0017466946737840772}, {'label': 'surprise', 'score': 0.001452988712117076}, {'label': 'gratitude', 'score': 0.0006464761681854725}, {'label': 'pride', 'score': 0.0005542495055124164}]]
jsonpath(json.loads(json.dumps(a)), '$..[?(@.label=="disappointment")].score')

[0.46669596433639526]

In [13]:
json.loads(json.dumps(a))

[[{'label': 'disappointment', 'score': 0.46669596433639526},
  {'label': 'sadness', 'score': 0.3984946608543396},
  {'label': 'annoyance', 'score': 0.06806609779596329},
  {'label': 'neutral', 'score': 0.05703018978238106},
  {'label': 'disapproval', 'score': 0.04423944652080536},
  {'label': 'nervousness', 'score': 0.014850739389657974},
  {'label': 'realization', 'score': 0.014059904962778091},
  {'label': 'approval', 'score': 0.011267454363405704},
  {'label': 'joy', 'score': 0.0063033816404640675},
  {'label': 'remorse', 'score': 0.006221489980816841},
  {'label': 'caring', 'score': 0.006029392126947641},
  {'label': 'embarrassment', 'score': 0.005265495739877224},
  {'label': 'anger', 'score': 0.004981442354619503},
  {'label': 'disgust', 'score': 0.004259037785232067},
  {'label': 'grief', 'score': 0.004002134781330824},
  {'label': 'confusion', 'score': 0.0033829279709607363},
  {'label': 'relief', 'score': 0.0031404944602400064},
  {'label': 'desire', 'score': 0.002827471587806

In [1]:
import pandas as pd
import os
datadir = '/mnt/data_disk/chu123/jyuzh/TangChuang/data/'
filename = 'Books'
file = pd.read_csv(os.path.join(datadir,f'{filename}_abstract.csv'))
file.head()

,asin,parent_asin,user_id,abstract
0,B09BGPFTDB,B09BGPFTDB,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0.474302
1,0593235657,0593235657,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0.394109
2,1782490671,1782490671,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0.487572
3,0593138228,0593138228,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0.470334
4,0823098079,0823098079,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,0.475147


In [2]:
file['concreteness'] = 1-file['abstract']
file.drop(columns=['abstract'], inplace=True)
file.to_csv(os.path.join(datadir,f'{filename}_concreteness.csv'), index=False)